# The Tidyverse

In [1]:
# Imports

library(NHANES)
library(dslabs)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   3.5.1     ✔ tibble    3.3.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


For serious data analysis, not vectors but data frames are the unit of analysis. So-called "tidy" data frames are used with an interrelated set of packages known as the "tidyverse" which was loaded all with the single line `library(tidyverse)` above.

The tidyverse approach to doing things will be introduced in the text but first there will be a little hands-on experience without too much theory. Some of the packages in the tidyverse include:

* dplyr (used for manipulating tibbles)
* purrr (extends functional programming for the tidyverse)
* ggplot2 (declarative graphing package based on Leland Wilkinson’s [*Grammar of Graphics*](https://en.wikipedia.org/wiki/Wilkinson%27s_Grammar_of_Graphics)
* readr (provides fast and easy reading of many forms of tabular data into R tibbles)

## Tidy data

Tidy data must meet the following three criteria:

1. Each row represents one observation.
2. Each column represents one of the variables present in each observation.
3. Each cell contains only one value.

The `murders` dataset is an example of tidy data:

In [2]:
head(murders)

,state,abb,region,population,total
,<chr>,<chr>,<fct>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135
2,Alaska,AK,West,710231,19
3,Arizona,AZ,West,6392017,232
4,Arkansas,AR,South,2915918,93
5,California,CA,West,37253956,1257
6,Colorado,CO,West,5029196,65


Here's another example of a tidy dataset, showing fertility rates for two countries over three years:

```
#>       country year fertility
#> 1     Germany 1960      2.41
#> 2 South Korea 1960      6.16
#> 3     Germany 1961      2.44
#> 4 South Korea 1961      5.99
#> 5     Germany 1962      2.47
#> 6 South Korea 1962      5.79
```

The original data were in the sort of "wide" format that one often sees for human convenience:

```
#>       country 1960 1961 1962
#> 1     Germany 2.41 2.44 2.47
#> 2 South Korea 6.16 5.99 5.79
```

Here there are multiple observations on each row as the `year` variable is distributed across multiple columns, violating tidy data principles. Typically tidy data will be in a "long" format, though less frequently it may be necessary to convert single observations split across multiple rows to a wide format. In any case, tidy data principles require an initial investment of discipline but, once this is done, the user will be freed to think about more important aspects of the analysis than the format of the data. Now for some exercises:

**Exercise 1**: Examine the built-in dataset `co2`. Which of the following is true:

1. `co2` is tidy data: it has one year for each row.
2. `co2` is not tidy: we need at least one column with a character vector.
3. `co2` is not tidy: it is a matrix instead of a data frame.
4. `co2` is not tidy: to be tidy we would have to wrangle it to have three columns (year, month and value), then each `co2` observation would have a row.

#4

**Exercise 2**: Examine the built-in dataset `ChickWeight`. Which of the following is true:

1. `ChickWeight` is not tidy: each chick has more than one row.
2. `ChickWeight` is tidy: each observation (a weight) is represented by one row. The chick from which this measurement came is one of the variables.
3. `ChickWeight` is not tidy: we are missing the year column.
4. `ChickWeight` is tidy: it is stored in a data frame.

#2

**Exercise 3**: Examine the built-in dataset `BOD`. Which of the following is true:

1. `BOD` is not tidy: it only has six rows.
2. `BOD` is not tidy: the first column is just an index.
3. `BOD` is tidy: each row is an observation with two values (time and demand)
4. `BOD` is tidy: all small datasets are tidy by definition.

#3

**Exercise 4**: Which of the following built-in datasets is tidy (you can pick more than one):

1. `BJsales`
2. `EuStockMarkets`
3. `DNase`
4. `Formaldehyde`
5. `Orange`
6. `UCBAdmissions`

#1, #3, #4, #5

## Refining data frames

The next section concerns the basics of dplyr, which provides staple data manipulation operations for with R data frames and tibbles. Among the operations provided by dplyr are:

* `mutate()`, which transforms columns that can then be added to existing data if desired.
* `filter()`, which returns row-based subsets of the data.
* `select()`, which subsets by columns rather than rows.

### Adding columns

Here's our first example using `mutate()` to add the per capita (per 100,000) homicide rate to `murders`. Note that merely running `mutate()` will *not* change the data structure in-place; rather the mutated copy must be assigned to the variable `murders`. Notice also that `mutate()`, like other dplyr "verbs", uses the [magic of non-standard evaluation in R](https://dplyr.tidyverse.org/articles/programming.html) to avoid having to spell everything out like one would have to with pandas in Python:

In [3]:
murders <- mutate(murders, rate = total / population * 10^5)
head(murders)

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424
2,Alaska,AK,West,710231,19,2.675186
3,Arizona,AZ,West,6392017,232,3.629527
4,Arkansas,AR,South,2915918,93,3.189390
5,California,CA,West,37253956,1257,3.374138
6,Colorado,CO,West,5029196,65,1.292453


### Row-wise subsetting

`filter()` does the aforementioned row-based subsetting. Here's an example with one Boolean condition and notice the same non-standard evaluation:

In [4]:
filter(murders, rate <= 0.71)

state,abb,region,population,total,rate
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
Hawaii,HI,West,1360301,7,0.5145920
Iowa,IA,North Central,3046355,21,0.6893484
New Hampshire,NH,Northeast,1316470,5,0.3798036
North Dakota,ND,North Central,672591,4,0.5947151
Vermont,VT,Northeast,625741,2,0.3196211


### Column-wise subsetting

Now for column-wise subsetting: `murders` has only six columns, but many datasets (especially before tidying) may have many more columns. Both for the purposes of human ease of use and to make machine analysis simpler, faster, and very possibly more effective (due to [including only the most important features](https://towardsdatascience.com/how-to-mitigate-overfitting-with-feature-selection-164897c0c3db/)), `select()` can be used to subset columns of a data frame or tibble. Here's a maybe not very useful but simple example based on `murders`:

In [5]:
new_df <- select(murders, state, region, rate)
filter(new_df, rate <= 0.71)

state,region,rate
<chr>,<fct>,<dbl>
Hawaii,West,0.5145920
Iowa,North Central,0.6893484
New Hampshire,Northeast,0.3798036
North Dakota,North Central,0.5947151
Vermont,Northeast,0.3196211


The Tidyverse package tidyselect offers a number of functions to select columns based on their content and not just by their full literal names. For example, here are the numeric columns of `murders` only:

In [6]:
new_df <- select(murders, where(is.numeric))
head(new_df)

,population,total,rate
,<dbl>,<dbl>,<dbl>
1,4779736,135,2.824424
2,710231,19,2.675186
3,6392017,232,3.629527
4,2915918,93,3.189390
5,37253956,1257,3.374138
6,5029196,65,1.292453


Here are a few other tidyselect functions and their descriptions:

* `starts_with()` (starts with a literal prefix).
* `ends_with()` (ends with a literal suffix).
* `contains()` (contains a literal string anywhere).
* `matches()` (matches a regular expression; there are nuances here such as the ability to use Tidyverse [stringr](https://stringr.tidyverse.org/articles/regular-expressions.html) objects).
* `num_range()` (matches strings like `wk01`, `wk02`, `wk03` etc.; [tidyselect documentation shows example use](https://tidyselect.r-lib.org/reference/starts_with.html)).

All of the above functions have an `ignore.case` parameter whose default argument is `TRUE` except for `num_range()`, which appears to be case-sensitive always. Anyway, here's a quick example of `starts_with()`:

In [7]:
new_df <- select(murders, starts_with('r'))
head(new_df)

,region,rate
,<fct>,<dbl>
1,South,2.824424
2,West,2.675186
3,West,3.629527
4,South,3.189390
5,West,3.374138
6,West,1.292453


### Transforming variables

Returning to the aforementioned `mutate()`, here are a few more examples, such as a $log_{10}(x)$ transformation:

In [8]:
head(mutate(murders, population = log10(population)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,6.679404,135,2.824424
2,Alaska,AK,West,5.851400,19,2.675186
3,Arizona,AZ,West,6.805638,232,3.629527
4,Arkansas,AR,South,6.464775,93,3.189390
5,California,CA,West,7.571172,1257,3.374138
6,Colorado,CO,West,6.701499,65,1.292453


The `across()` function allows several ways to use `mutate()` on multiple columns. Here it is used with two literals:

In [9]:
head(mutate(murders, across(c(population, total), log10)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,6.679404,2.130334,2.824424
2,Alaska,AK,West,5.851400,1.278754,2.675186
3,Arizona,AZ,West,6.805638,2.365488,3.629527
4,Arkansas,AR,South,6.464775,1.968483,3.189390
5,California,CA,West,7.571172,3.099335,3.374138
6,Colorado,CO,West,6.701499,1.812913,1.292453


Predicate functions can also be used with `across()`. Here, all numeric variables are transformed with `log10()`:

In [10]:
head(mutate(murders, across(where(is.numeric), log10)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,6.679404,2.130334,0.4509299
2,Alaska,AK,West,5.851400,1.278754,0.4273540
3,Arizona,AZ,West,6.805638,2.365488,0.5598501
4,Arkansas,AR,South,6.464775,1.968483,0.5037076
5,California,CA,West,7.571172,3.099335,0.5281629
6,Colorado,CO,West,6.701499,1.812913,0.1114148


And here, character variables are lowercased:

In [11]:
head(mutate(murders, across(where(is.character), tolower)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,alabama,al,South,4779736,135,2.824424
2,alaska,ak,West,710231,19,2.675186
3,arizona,az,West,6392017,232,3.629527
4,arkansas,ar,South,2915918,93,3.189390
5,california,ca,West,37253956,1257,3.374138
6,colorado,co,West,5029196,65,1.292453


Now for a few more exercises.

**Exercise 5**: You can add columns using the dplyr function `mutate()`. This function is aware of the column names and inside the function you can call them unquoted:

```r
murders <- mutate(murders, population_in_millions = population/10^6)
```

We can write `population` rather than `murders$population`. The function `mutate()` knows we are grabbing columns from `murders`.

Use the function `mutate()` to add a `murders` column named `rate` with the per 100,000 murder rate \[similar to\] the example code above. Make sure you redefine `murders` as done in the example code above (murders <- \[your code\]) so we can keep using this variable.

In [12]:
murders <- mutate(murders, rate = total / population * 10^5)
head(murders)

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424
2,Alaska,AK,West,710231,19,2.675186
3,Arizona,AZ,West,6392017,232,3.629527
4,Arkansas,AR,South,2915918,93,3.189390
5,California,CA,West,37253956,1257,3.374138
6,Colorado,CO,West,5029196,65,1.292453


**Exercise 6**: If `rank(x)` gives you the ranks of `x` from lowest to highest, `rank(-x)` gives you the ranks from highest to lowest. Use the function `mutate()` to add a column `rank` containing the rank of murder rate from highest to lowest. Make sure you redefine `murders` so we can keep using this variable.

In [13]:
murders <- mutate(murders, rank = rank(-rate))
head(murders)

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424,23
2,Alaska,AK,West,710231,19,2.675186,27
3,Arizona,AZ,West,6392017,232,3.629527,10
4,Arkansas,AR,South,2915918,93,3.189390,17
5,California,CA,West,37253956,1257,3.374138,14
6,Colorado,CO,West,5029196,65,1.292453,38


N.b.: `arrange()` will come up later but it can absolutely be used here:

In [14]:
arrange(murders, rank)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
District of Columbia,DC,South,601723,99,16.4527532,1
Louisiana,LA,South,4533372,351,7.7425810,2
Missouri,MO,North Central,5988927,321,5.3598917,3
Maryland,MD,South,5773552,293,5.0748655,4
South Carolina,SC,South,4625364,207,4.4753235,5
Delaware,DE,South,897934,38,4.2319369,6
Michigan,MI,North Central,9883640,413,4.1786225,7
Mississippi,MS,South,2967297,120,4.0440846,8
Georgia,GA,South,9920000,376,3.7903226,9


**Exercise 7**: With dplyr, we can use `select()` to show only certain columns. For example, with this code we would only show the states and population sizes:

```r
select(murders, state, population)
```

\[The following also works:\]

```r
select(murders, c(state, population))
```

Use `select()` to show the state names and abbreviations in `murders`. Do not redefine `murders`, just show the results.

In [15]:
head(select(murders, c(state, abb)))

,state,abb
,<chr>,<chr>
1,Alabama,AL
2,Alaska,AK
3,Arizona,AZ
4,Arkansas,AR
5,California,CA
6,Colorado,CO


**Exercise 8**: The dplyr function `filter()` is used to choose specific rows of the data frame to keep. Unlike `select()`, which is for columns, `filter()` is for rows. For example, you can show just the New York row like this:

```r
filter(murders, state == 'New York')
```

You can use other logical vectors to filter rows.

Use `filter()` to show the top 5 states with the highest murder rates. From here on, do not change the `murders` dataset, just show the result. Remember that you can `filter()` based on the `rank` column. \[I will be sorting using `arrange()` again.\]

In [16]:
filter(murders, rank <= 5) |> arrange(rank)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
District of Columbia,DC,South,601723,99,16.452753,1
Louisiana,LA,South,4533372,351,7.742581,2
Missouri,MO,North Central,5988927,321,5.359892,3
Maryland,MD,South,5773552,293,5.074866,4
South Carolina,SC,South,4625364,207,4.475323,5


**Exercise 9**: We can remove rows using the `!=` operator. For example, to remove Florida, we would do this:

```r
no_florida <- filter(murders, state != 'Florida')
```

Create a new data frame called `no_south` that removes states from the South region. How many states are in this category? You can use the function `nrow()` for this.

In [17]:
no_south <- filter(murders, region != 'South')
no_south

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Illinois,IL,North Central,12830632,364,2.8369608,22
Indiana,IN,North Central,6483802,142,2.1900730,31


In [18]:
nrow(no_south)

[1] 34

**Exercise 10**: We can also use `%in%` to filter with dplyr. You can therefore see the data from New York and Texas like this:

```r
filter(murders, state %in% c('New York', 'Texas'))
```

Create a new data frame called `murders_nw` with only the states from the Northeast and the West. How many states are in this category?

In [19]:
murders_nw <- filter(murders, region %in% c('Northeast', 'West'))
murders_nw

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Maine,ME,Northeast,1328361,11,0.8280881,44
Massachusetts,MA,Northeast,6547629,118,1.8021791,32


In [20]:
nrow(murders_nw)

[1] 22

**Exercise 11**: Suppose you want to live in the Northeast or West and want the murder rate to be less than 1. We want to see the data for the states satisfying these options. Note that you can use logical operators like `&` with `filter()`. Here is an example in which we filter to keep only small states in the Northeast region.

```r
filter(murders, population < 5000000 & region == 'Northeast')
```

Make sure `murders` has been defined with `rate` and `rank` and still has all states. Create a table called `my_states` that contains rows for states satisfying both the conditions: it is in the Northeast or West and the murder rate is less than 1. Use `select()` to show only the state name, the rate, and the rank.

In [21]:
my_states <- filter(murders, region %in% c('Northeast', 'West') & rate < 1)
my_states

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Maine,ME,Northeast,1328361,11,0.8280881,44
New Hampshire,NH,Northeast,1316470,5,0.3798036,50
Oregon,OR,West,3831074,36,0.9396843,42
Utah,UT,West,2763885,22,0.7959810,45
Vermont,VT,Northeast,625741,2,0.3196211,51
Wyoming,WY,West,563626,5,0.8871131,43


## The pipe

Initially in the third-party package magrittr, using the operator `%>%`, and now since R 4.1.0 with the operator in the base package `|>`, one can construct "pipelines" of functions that are more readable than the alternative. What `|>` does is give the left-hand operand as the first argument to a function that is given as the right-hand operand. Since that's a mouthful, it may be better just to look at an example using `select()` and `filter()`:

In [22]:
murders |> select(state, region, rate) |> filter(rate <= 0.71)

state,region,rate
<chr>,<fct>,<dbl>
Hawaii,West,0.5145920
Iowa,North Central,0.6893484
New Hampshire,Northeast,0.3798036
North Dakota,North Central,0.5947151
Vermont,Northeast,0.3196211


Now consider the alternative:

In [23]:
filter(select(murders, state, region, rate), rate <= 0.71)

state,region,rate
<chr>,<fct>,<dbl>
Hawaii,West,0.5145920
Iowa,North Central,0.6893484
New Hampshire,Northeast,0.3798036
North Dakota,North Central,0.5947151
Vermont,Northeast,0.3196211


Using the pipeline operator `|>` clearly exposes the order of execution and is much more legible, particularly with longer pipelines. Note that other arguments, including keyword arguments, can be specified in these pipelines. Here's a fairly trivial example:

In [24]:
16 |> sqrt() |> log(base = 2)

[1] 2

## Summarizing data

Having gotten the basics of pipelines out of the way, it is now time to address a very important part of data exploration and analysis, and that's computing summary statistics either for an entire dataset or for a [partition](https://math.libretexts.org/Bookshelves/Combinatorics_and_Discrete_Mathematics/Applied_Discrete_Structures_(Doerr_and_Levasseur)/02%3A_Combinatorics/2.03%3A_Partitions_of_Sets_and_the_Law_of_Addition) of subsets of the data. The Tidyverse functions used for doing so are `summarise()` (alternatively `summarize()` which I'll be using because of muh freedoms) and `group_by()`.

### The `summarize()` function

First let's look at the basic use of `summarize()`. Here are the average and standard deviation of females in the `heights` dataset:

In [25]:
s <- heights |> filter(sex == 'Female') |> summarize(mean(height), sd(height))
s

mean(height),sd(height)
<dbl>,<dbl>
64.93942,3.760656


It's also possible to specify the variable names:

In [26]:
s <- heights |> filter(sex == 'Female') |> summarize(average = mean(height), standard_deviation = sd(height))
s

average,standard_deviation
<dbl>,<dbl>
64.93942,3.760656


The result `s` here is a data frame and so the accessor `$` works on it:

In [27]:
s$average

[1] 64.93942

In [28]:
s$standard_deviation

[1] 3.760656

Here's another example of `summarize()`, calculating the US national homicide rate the *wrong* way:

In [29]:
murders |> summarize(rate = mean(rate))

rate
<dbl>
2.779125


In the above cell, small states and large states were treated equally. The *true* US homicide rate (per 100,000) is the sum of all the homicides divided by the population and then of course multiplied by 100,000:

In [30]:
us_murder_rate <- murders |> summarize(rate = sum(total) / sum(population) * 10^5)
us_murder_rate

rate
<dbl>
3.034555


### Multiple summaries

Suppose we want the median, minimum and maximum of `heights`. These values can be calculated from the data using `summarize()` like so:

In [31]:
heights |> summarize(median = median(height), min = min(height), max = max(height))

median,min,max
<dbl>,<dbl>,<dbl>
68.5,50,82.67717


It's also possible to render the same values using the `quantile()` function.

In [32]:
quantile(heights$height, c(0.5, 0, 1))

50%       0%     100% 
68.50000 50.00000 82.67717

However it's not possible to use `summarize()` with `quantile()` because `quantile()` expects exactly one row of output for any of the returned columns. Attempting to do otherwise will raise an error and suggest you use `reframe()` which will now be done:

In [33]:
heights |> reframe(quantiles = quantile(height, c(0.5, 0, 1)))

quantiles
<dbl>
68.50000
50.00000
82.67717


Oops, but that's several rows rather than columns and they're unlabeled! Here's the (clunky) workaround supplied by the book:

In [34]:
median_min_max <- function(x) {
  qs <- quantile(x, c(0.5, 0, 1))
  data.frame(median = qs[1], min = qs[2], max = qs[3])
}

heights |> summarize(median_min_max(height))

median,min,max
<dbl>,<dbl>,<dbl>
68.5,50,82.67717


### Group then summarize with `group_by()`

The previous examples assumed that summary statistics were desired for the *entire* dataset. The dplyr function `group_by()` will be introduced now, which enables summary statistics for groups that partition the dataset. Here's what `group_by()` looks like on its own:

In [35]:
heights |> group_by(sex) |> head() |> print()

# A tibble: 6 × 2
# Groups:   sex [2]
  sex    height
  <fct>   <dbl>
1 Male       75
2 Male       70
3 Male       68
4 Male       74
5 Male       61
6 Female     65


(Note that I used `print()` for clarity of output.) The output is now a tibble and `# Groups: sex [2]` appears. dplyr functions will behave differently with the grouped tibble. Here are the summary statistics of mean and standard deviation for the separate groups of `Male` and `Female`:

In [36]:
heights |> group_by(sex) |> summarize(average = mean(height), standard_deviation = sd(height))

sex,average,standard_deviation
<fct>,<dbl>,<dbl>
Female,64.93942,3.760656
Male,69.31475,3.611024


Here's another example with summary statistics of the median, minimum and maximum for each region in `murders`:

In [37]:
murders |> group_by(region) |> summarize(median_min_max(rate))

region,median,min,max
<fct>,<dbl>,<dbl>,<dbl>
Northeast,1.802179,0.3196211,3.597751
South,3.398069,1.4571013,16.452753
North Central,1.971105,0.5947151,5.359892
West,1.292453,0.5145920,3.629527


### Extracting variables with `pull()`

Objects returned by `summarize()` are always a form of data frame, even if they consist of only one value:

In [38]:
class(us_murder_rate)

[1] "data.frame"

The function `pull()` can be used to change that!

In [39]:
us_murder_rate |> pull()

[1] 3.034555

In [40]:
us_murder_rate |> pull() |> class()

[1] "numeric"

## Sorting

The last thing to be covered before the next exercises will be sorting with the dplyr function `arrange()`. Here's `murders` sorted by `population`, ascending:

In [41]:
murders |> arrange(population) |> head()

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Wyoming,WY,West,563626,5,0.8871131,43
2,District of Columbia,DC,South,601723,99,16.4527532,1
3,Vermont,VT,Northeast,625741,2,0.3196211,51
4,North Dakota,ND,North Central,672591,4,0.5947151,48
5,Alaska,AK,West,710231,19,2.6751860,27
6,South Dakota,SD,North Central,814180,8,0.9825837,41


Here's the same thing but with *descending* order:

In [42]:
murders |> arrange(desc(population)) |> head()

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,California,CA,West,37253956,1257,3.374138,14
2,Texas,TX,South,25145561,805,3.201360,16
3,Florida,FL,South,19687653,669,3.398069,13
4,New York,NY,Northeast,19378102,517,2.667960,29
5,Illinois,IL,North Central,12830632,364,2.836961,22
6,Pennsylvania,PA,Northeast,12702379,457,3.597751,11


### Nested sorting

It is also possible to sort by *multiple* columns:

In [43]:
murders |> arrange(region, rate) |> head(n = 15)

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Vermont,VT,Northeast,625741,2,0.3196211,51
2,New Hampshire,NH,Northeast,1316470,5,0.3798036,50
3,Maine,ME,Northeast,1328361,11,0.8280881,44
4,Rhode Island,RI,Northeast,1052567,16,1.5200933,35
5,Massachusetts,MA,Northeast,6547629,118,1.8021791,32
6,New York,NY,Northeast,19378102,517,2.6679599,29
7,Connecticut,CT,Northeast,3574097,97,2.7139722,25
8,New Jersey,NJ,Northeast,8791894,246,2.7980319,24
9,Pennsylvania,PA,Northeast,12702379,457,3.5977513,11


### The top $n$

There are also the convenience functions `slice_max()` and `slice_min()` which return, respectively the top or bottom $n$ (apparently by default 1) rows in a data frame or tibble according to a column specified by the user:

In [44]:
murders |> slice_max(rate, n = 5)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
District of Columbia,DC,South,601723,99,16.452753,1
Louisiana,LA,South,4533372,351,7.742581,2
Missouri,MO,North Central,5988927,321,5.359892,3
Maryland,MD,South,5773552,293,5.074866,4
South Carolina,SC,South,4625364,207,4.475323,5


In [45]:
murders |> slice_min(rate, n = 5)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Vermont,VT,Northeast,625741,2,0.3196211,51
New Hampshire,NH,Northeast,1316470,5,0.3798036,50
Hawaii,HI,West,1360301,7,0.5145920,49
North Dakota,ND,North Central,672591,4,0.5947151,48
Iowa,IA,North Central,3046355,21,0.6893484,47


The following code is equivalent to the last two code blocks:

In [46]:
murders |> arrange(desc(rate)) |> head(n = 5)

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,District of Columbia,DC,South,601723,99,16.452753,1
2,Louisiana,LA,South,4533372,351,7.742581,2
3,Missouri,MO,North Central,5988927,321,5.359892,3
4,Maryland,MD,South,5773552,293,5.074866,4
5,South Carolina,SC,South,4625364,207,4.475323,5


In [47]:
murders |> arrange(rate) |> head(n = 5)

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Vermont,VT,Northeast,625741,2,0.3196211,51
2,New Hampshire,NH,Northeast,1316470,5,0.3798036,50
3,Hawaii,HI,West,1360301,7,0.5145920,49
4,North Dakota,ND,North Central,672591,4,0.5947151,48
5,Iowa,IA,North Central,3046355,21,0.6893484,47


Now for more exercises:

**Exercise 12**: The pipe `|>` can be used to perform operations sequentially without having to define intermediate objects. Start by redefining murders to include rate and rank. \[Already done.\]

In the solution to the previous exercise, we did the following:

```r
my_states <- filter(murders, region %in% c('Northeast', 'West') & rate < 1)
select(my_states, state, rate, rank)
```

The pipe `|>` permits us to perform both operations sequentially without having to define an intermediate variable `my_states`. We therefore could have mutated and selected in the same line like this:

```r
mutate(murders, rate = total/population * 10^5, 
                rank = rank(-rate)) |>
select(state, rate, rank)
```

Notice that `select()` no longer has a data frame as the first \[explicit\] argument. The first argument is assumed to be the result of the operation conducted right before the `|>`.

Repeat the previous exercise, but now instead of creating a new object, show the result and only include the `state`, `rate`, and `rank` columns. Use a pipe `|>` to do this in just one line.

In [48]:
murders |>
    mutate(rate = total / population * 10^5,
           rank = rank(-rate)) |>
    filter(region %in% c('Northeast', 'West') & rate < 1) |>
    select(state, rate, rank)

state,rate,rank
<chr>,<dbl>,<dbl>
Hawaii,0.5145920,49
Idaho,0.7655102,46
Maine,0.8280881,44
New Hampshire,0.3798036,50
Oregon,0.9396843,42
Utah,0.7959810,45
Vermont,0.3196211,51
Wyoming,0.8871131,43


**Exercise 13**: For exercises 13-19, we will be using the data from the survey collected by the United States National Center for Health Statistics (NCHS). This center has conducted a series of health and nutrition surveys since the 1960’s. Starting in 1999, about 5,000 individuals of all ages have been interviewed every year and they complete the health examination component of the survey. Part of the data is made available via the NHANES package. Once you install the NHANES package, you can load the data like this:

```r
library(NHANES)
```

The NHANES data has many missing values. The `mean()` and `sd()` functions in R will return `NA` if any of the entries of the input vector is an `NA`. Here is an example:

```r
library(dslabs)
mean(na_example)
#> [1] NA
sd(na_example)
#> [1] NA
```

To ignore the `NA`s we can use the `na.rm` argument:

```r
mean(na_example, na.rm = TRUE)
#> [1] 2.3
sd(na_example, na.rm = TRUE)
#> [1] 1.22
```

Let's now explore the NHANES data.

We will provide some basic facts about blood pressure. First let's select a group to set the standard. We will use 20-to-29-year-old females. `AgeDecade` is a categorical variable with these ages. Note that the category is coded like `" 20-29"`, with a space in front! What is the average and standard deviation of systolic blood pressure as saved in the `BPSysAve` variable? Save it to a variable called `ref`.

Hint: Use `filter()` and `summarize()` and use the `na.rm = TRUE` argument when computing the average and standard deviation. You can also filter the `NA` values using `filter()`. \[And that is actually a fantastic idea.\]

In [49]:
ref <- NHANES |>
    filter(Gender == 'female' & AgeDecade == ' 20-29' & !is.na(BPSysAve)) |>
    summarize(mean = mean(BPSysAve), sd = sd(BPSysAve))
ref

mean,sd
<dbl>,<dbl>
108.4224,10.14668


**Exercise 14**: Using a pipe, assign the average to a numeric variable `ref_avg`. Hint: Use the code from the previous exercise and then `pull()`.

In [50]:
ref_avg <- ref |> pull(mean)
ref_avg

[1] 108.4224

**Exercise 15**: Now report the min and max values for the same group.

In [51]:
NHANES |>
    filter(Gender == 'female' & AgeDecade == ' 20-29' & !is.na(BPSysAve)) |>
    summarize(min = min(BPSysAve), max = max(BPSysAve))

min,max
<int>,<int>
84,179


**Exercise 16**: Compute the average and standard deviation for females, but for each age group separately rather than a selected decade as in exercise 13. Note that the age groups are defined by `AgeDecade`. Hint: rather than filtering by `Age` and `Gender`, filter by `Gender` and then use `group_by()`.

In [52]:
NHANES |>
    filter(Gender == 'female' & !is.na(BPSysAve)) |>
    group_by(AgeDecade) |>
    summarize(mean = mean(BPSysAve), sd = sd(BPSysAve))

AgeDecade,mean,sd
<fct>,<dbl>,<dbl>
0-9,99.95041,9.071798
10-19,104.27466,9.461431
20-29,108.42243,10.146681
30-39,111.25512,12.314790
40-49,115.49385,14.530054
50-59,121.84245,16.179333
60-69,127.17787,17.125713
70+,133.51652,19.841781
NA,141.54839,22.908521


**Exercise 17**: Repeat exercise 16 for males.

In [53]:
NHANES |>
    filter(Gender == 'male' & !is.na(BPSysAve)) |>
    group_by(AgeDecade) |>
    summarize(mean = mean(BPSysAve), sd = sd(BPSysAve))

AgeDecade,mean,sd
<fct>,<dbl>,<dbl>
0-9,97.41912,8.317367
10-19,109.59789,11.227769
20-29,117.85084,11.274795
30-39,119.40063,12.306656
40-49,120.78390,13.968338
50-59,125.75000,17.760536
60-69,126.88578,17.478117
70+,130.20172,18.657475
NA,136.40000,23.534731


**Exercise 18**: For males between the ages of 40-49, compare systolic blood pressure across race as reported in the `Race1` variable. Order the resulting table from lowest to highest average systolic blood pressure.

In [54]:
NHANES |>
    filter(Gender == 'male' & AgeDecade == ' 40-49' & !is.na(BPSysAve)) |>
    group_by(Race1) |>
    summarize(mean = mean(BPSysAve)) |>
    arrange(mean)

Race1,mean
<fct>,<dbl>
White,119.9188
Other,120.4000
Hispanic,121.6098
Mexican,121.8500
Black,125.8387


**Exercise 19**: Load the `murders` dataset. Which of the following is true?

1. `murders` is in tidy format and is stored in a tibble.
2. `murders` is in tidy format and is stored in a data frame.
3. `murders` is not in tidy format and is stored in a tibble.
4. `murders` is not in tidy format and is stored in a data frame.

In [55]:
head(murders)

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424,23
2,Alaska,AK,West,710231,19,2.675186,27
3,Arizona,AZ,West,6392017,232,3.629527,10
4,Arkansas,AR,South,2915918,93,3.189390,17
5,California,CA,West,37253956,1257,3.374138,14
6,Colorado,CO,West,5029196,65,1.292453,38


`murders` is tidy.

In [56]:
class(murders)

[1] "data.frame"

`murders` is a data frame.

## Tibbles

What exactly is a tibble anyway? It's an enhanced version of R's built-in data frame meant for use with Tidyverse packages. One of these features is the `group_by()` functionality that data frames don't exactly have. `group_by()` and `summarize()` always return tibbles and `group_by()` specifically returns a "grouped_df":

In [57]:
murders |> group_by(region) |> class()

[1] "grouped_df" "tbl_df"     "tbl"        "data.frame"

Some Tidyverse functions respect the integrity of data frames; `select()`, `filter()`, `mutate()` and `arrange()` return data frames if they are given data frames. However, tibbles are preferred in the Tidyverse and Tidyverse functions that return data from elsewhere rather than working with an existing data frame return tibbles. More of this will be seen in chapter 6.

## Tibbles versus data frames

Anyway, here are some key differences from data frames:

**Tibbles display better**: Compare printing `murders` as the original data frame and as a tibble. (Default Jupyter behavior has been suppressed with `print()` so this is what you will see in a regular R console or script output.)

In [58]:
print(murders)

                  state abb        region population total       rate rank
1               Alabama  AL         South    4779736   135  2.8244238   23
2                Alaska  AK          West     710231    19  2.6751860   27
3               Arizona  AZ          West    6392017   232  3.6295273   10
4              Arkansas  AR         South    2915918    93  3.1893901   17
5            California  CA          West   37253956  1257  3.3741383   14
6              Colorado  CO          West    5029196    65  1.2924531   38
7           Connecticut  CT     Northeast    3574097    97  2.7139722   25
8              Delaware  DE         South     897934    38  4.2319369    6
9  District of Columbia  DC         South     601723    99 16.4527532    1
10              Florida  FL         South   19687653   669  3.3980688   13
11              Georgia  GA         South    9920000   376  3.7903226    9
12               Hawaii  HI          West    1360301     7  0.5145920   49
13                Idaho  

In [59]:
murders |> as_tibble() |> print()

# A tibble: 51 × 7
   state                abb   region    population total  rate  rank
   <chr>                <chr> <fct>          <dbl> <dbl> <dbl> <dbl>
 1 Alabama              AL    South        4779736   135  2.82    23
 2 Alaska               AK    West          710231    19  2.68    27
 3 Arizona              AZ    West         6392017   232  3.63    10
 4 Arkansas             AR    South        2915918    93  3.19    17
 5 California           CA    West        37253956  1257  3.37    14
 6 Colorado             CO    West         5029196    65  1.29    38
 7 Connecticut          CT    Northeast    3574097    97  2.71    25
 8 Delaware             DE    South         897934    38  4.23     6
 9 District of Columbia DC    South         601723    99 16.5      1
10 Florida              FL    South       19687653   669  3.40    13
# ℹ 41 more rows


**Tibbles subset more intelligently**: If you subset the `murders` data frame with `[`, you may get an object that is not a data frame. Here you will indeed get a data frame:

In [60]:
murders[1:5, 1:4]

,state,abb,region,population
,<chr>,<chr>,<fct>,<dbl>
1,Alabama,AL,South,4779736
2,Alaska,AK,West,710231
3,Arizona,AZ,West,6392017
4,Arkansas,AR,South,2915918
5,California,CA,West,37253956


In [61]:
class(murders[1:5, 1:4])

[1] "data.frame"

Even one row can be extracted and remain a data frame:

In [62]:
murders[1, ]

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424,23


In [63]:
class(murders[1, ])

[1] "data.frame"

...but upon extracting a single column with `[`, you will not:

In [64]:
murders[, 1]

[1] "Alabama"              "Alaska"               "Arizona"             
 [4] "Arkansas"             "California"           "Colorado"            
 [7] "Connecticut"          "Delaware"             "District of Columbia"
[10] "Florida"              "Georgia"              "Hawaii"              
[13] "Idaho"                "Illinois"             "Indiana"             
[16] "Iowa"                 "Kansas"               "Kentucky"            
[19] "Louisiana"            "Maine"                "Maryland"            
[22] "Massachusetts"        "Michigan"             "Minnesota"           
[25] "Mississippi"          "Missouri"             "Montana"             
[28] "Nebraska"             "Nevada"               "New Hampshire"       
[31] "New Jersey"           "New Mexico"           "New York"            
[34] "North Carolina"       "North Dakota"         "Ohio"                
[37] "Oklahoma"             "Oregon"               "Pennsylvania"        
[40] "Rhode Island"         "South Carolina"       "South Dakota"        
[43] "Tennessee"            "Texas"                "Utah"                
[46] "Vermont"              "Virginia"             "Washington"          
[49] "West Virginia"        "Wisconsin"            "Wyoming"

In [65]:
class(murders[, 1])

[1] "character"

With tibbles this does not happen, which is important because Tidyverse functions require either data frames or tibbles as input:

In [66]:
(murders |> as_tibble())[, 1]

state
<chr>
Alabama
Alaska
Arizona
Arkansas
California
Colorado
Connecticut
Delaware
District of Columbia


With tibbles, if you want to extract a column as a vector, it will be necessary to use the `$` operator:

In [67]:
(murders |> as_tibble())$state

[1] "Alabama"              "Alaska"               "Arizona"             
 [4] "Arkansas"             "California"           "Colorado"            
 [7] "Connecticut"          "Delaware"             "District of Columbia"
[10] "Florida"              "Georgia"              "Hawaii"              
[13] "Idaho"                "Illinois"             "Indiana"             
[16] "Iowa"                 "Kansas"               "Kentucky"            
[19] "Louisiana"            "Maine"                "Maryland"            
[22] "Massachusetts"        "Michigan"             "Minnesota"           
[25] "Mississippi"          "Missouri"             "Montana"             
[28] "Nebraska"             "Nevada"               "New Hampshire"       
[31] "New Jersey"           "New Mexico"           "New York"            
[34] "North Carolina"       "North Dakota"         "Ohio"                
[37] "Oklahoma"             "Oregon"               "Pennsylvania"        
[40] "Rhode Island"         "South Carolina"       "South Dakota"        
[43] "Tennessee"            "Texas"                "Utah"                
[46] "Vermont"              "Virginia"             "Washington"          
[49] "West Virginia"        "Wisconsin"            "Wyoming"

In [68]:
class((murders |> as_tibble())$state)

[1] "character"

**Tibbles also include more diagnostic information than data frames**: Consider accessing the nonexistent column `Population` in the data frame `murders` (and remember that case matters):

In [69]:
murders$Population

NULL

A tibble will still return `NULL` but emit a warning:

In [70]:
as_tibble(murders)$Population

Warning message:
“Unknown or uninitialised column: `Population`.”


NULL

**Tibbles can contain more types of objects than data frames**: Data frames are stuck with numeric, character and logical objects. Tibbles can include columns with lists and even functions:

In [71]:
# JupyterLab won't display this without print()
tibble(id = 1:3, func = c(mean, median, sd)) |> print()

# A tibble: 3 × 2
     id func  
  <int> <list>
1     1 <fn>  
2     2 <fn>  
3     3 <fn>  


**Tibbles can be grouped**: As has already been mentioned, tibbles can be grouped with `group_by()` and Tidyverse functions (particularly `summarize()`) will take that grouping into consideration.

### Creating tibbles

As shown earlier, tibbles can be constructed from data supplied in code with the `tibble()` function. It's not super common but it should be known about:

In [72]:
grades <- tibble(names = c('John', 'Juan', 'Jean', 'Yao'),
                 exam_1 = c(95, 80, 90, 85),
                 exam_2 = c(90, 85, 85, 90))
grades

names,exam_1,exam_2
<chr>,<dbl>,<dbl>
John,95,90
Juan,80,85
Jean,90,85
Yao,85,90


As also mentioned the same can be done with data frames:

In [73]:
grades <- data.frame(names = c('John', 'Juan', 'Jean', 'Yao'),
                     exam_1 = c(95, 80, 90, 85),
                     exam_2 = c(90, 85, 85, 90))
grades

names,exam_1,exam_2
<chr>,<dbl>,<dbl>
John,95,90
Juan,80,85
Jean,90,85
Yao,85,90


Recall that a data frame can be converted to a tibble easily:

In [74]:
grades |> as_tibble() |> class()

[1] "tbl_df"     "tbl"        "data.frame"

## The placeholder

Here I'm pretty much just going to crib from the text:

>One of the advantages of using the pipe `|>` is that we do not have to keep naming new objects as we manipulate the data frame \[or tibble\]. The object on the left-hand side of the pipe is used as the first argument of the function on the right-hand side of the pipe. But what if we want to pass it as argument to the right-hand side function that is not the first? The answer is the placeholder operator `_` (for the `%>%` pipe the placeholder is `.`). Below is a simple example that passes the base argument to the `log()` function. The following three are equivalent:

In [75]:
log(8, base = 2)

[1] 3

In [76]:
2 |> log(8, base = _)

[1] 3

In [77]:
2 %>% log(8, base = .)

[1] 3

## The purrr package

Back in chapter 3, it was shown how to use `sapply()` to map a function onto the contents of a vector like so: 